In [ ]:
code = 'JADE_LIZARD'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def JADE_LIZARD(bt, start_time, end_time, variant, short_om, wing_width, sl, dte1re, dte2re, dte3re, dte4re, dte5re):
    try:
        variant = str(variant).upper().strip()
        if variant not in ('JADE', 'REVERSE'): return None

        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)

        def run_cycle(from_dt):

            short_ce_scrip, short_pe_scrip, _, _, future_price, entry_dt = bt.get_strike(from_dt, end_dt, om=short_om)
            if short_ce_scrip is None: return None

            step = int(wing_width) * bt.gap

            ### JADE = short put + short call spread (no upside risk) | REVERSE = short call + short put spread (no downside risk)
            if variant == 'JADE':
                long_scrip = f"{get_strike(short_ce_scrip) + step}CE"
                if get_strike(long_scrip) <= get_strike(short_ce_scrip): return None
            else:
                long_scrip = f"{get_strike(short_pe_scrip) - step}PE"
                if get_strike(long_scrip) >= get_strike(short_pe_scrip): return None

            short_ce_data = bt.get_single_leg_data(entry_dt, end_dt, short_ce_scrip)
            short_pe_data = bt.get_single_leg_data(entry_dt, end_dt, short_pe_scrip)
            long_data = bt.get_single_leg_data(entry_dt, end_dt, long_scrip)

            common_dt = np.intersect1d(short_ce_data['date_time'].values, short_pe_data['date_time'].values)
            common_dt = np.intersect1d(common_dt, long_data['date_time'].values)

            if len(common_dt) == 0:
                return None

            short_ce_data = short_ce_data[np.isin(short_ce_data['date_time'].values, common_dt)]
            short_pe_data = short_pe_data[np.isin(short_pe_data['date_time'].values, common_dt)]
            long_data = long_data[np.isin(long_data['date_time'].values, common_dt)]

            cycle_entry_time = short_ce_data['date_time'].iloc[0]

            short_ce_price = short_ce_data['close'].iloc[0]
            short_pe_price = short_pe_data['close'].iloc[0]
            long_price = long_data['close'].iloc[0]

            credit = (short_ce_price + short_pe_price) - long_price
            if credit <= 0:
                return None

            ### defining jade lizard condition - credit covers the spread width, so that tail carries no risk
            no_tail_risk = bool(credit >= step)

            close_value_list = ((short_ce_data['close'].values + short_pe_data['close'].values) - long_data['close'].values).tolist()
            eod_value = close_value_list[-1]

            slipage = bt.Cal_slipage(short_ce_price + short_pe_price + long_price)
            no_sl_pnl = round((credit - eod_value) - slipage, 2)

            sl_hit, sl_time = False, ''

            try:
                sl_index = next(i for i, ele in enumerate(close_value_list) if ele >= credit * (1 + (sl/100)))
                sl_time = short_ce_data['date_time'].iloc[sl_index]
                sl_pnl = round((credit - close_value_list[sl_index]) - slipage, 2)
                sl_hit = True
            except StopIteration:
                sl_pnl = no_sl_pnl

            legs = f"({short_ce_scrip}, {short_pe_scrip}, {long_scrip})"
            return [cycle_entry_time, legs, credit, no_tail_risk, sl_hit, sl_time, sl_pnl], future_price

        cycle = run_cycle(start_dt)
        if cycle is None: return None
        trade, future_price = cycle

        entry_time = trade[0]
        trades = [trade]
        exit_time = trade[5] if trade[4] else ''

        re_entries_left = {1: dte1re, 2: dte2re, 3: dte3re, 4: dte4re, 5: dte5re}
        blank_slot = ['', '', '', False, False, '', 0]

        re_trades = []
        for re_no in range(max_re):

            if exit_time and (exit_time < end_dt - datetime.timedelta(minutes=5)):

                rdte = int(dte_file.loc[pd.to_datetime(exit_time.date()), bt.index])

                ### budget spent for this dte - park the re-entry near the close so it carries into the next dte
                if re_entries_left.get(rdte, 0) == 0:
                    if rdte == 1:
                        exit_time = ''
                        re_trades.extend(blank_slot)
                        continue

                    exit_time = max(exit_time, (datetime.datetime.combine(exit_time.date(), bt.meta_end_time) - datetime.timedelta(minutes=15)))
                else:
                    re_entries_left[rdte] -= 1

                cycle = run_cycle(exit_time)
                if cycle is None:
                    exit_time = ''
                    re_trades.extend(blank_slot)
                    continue

                trade, _ = cycle
                trades.append(trade)
                re_trades.extend(trade)
                exit_time = trade[5] if trade[4] else ''
            else:
                re_trades.extend(blank_slot)

        total_pnl = round(sum(t[-1] for t in trades), 2)
        re_count = len(trades) - 1

        dte_pnl = {5: 0, 4: 0, 3: 0, 2: 0, 1: 0}
        for t in trades:
            tdte = int(dte_file.loc[pd.to_datetime(t[0].date()), bt.index])
            if tdte in dte_pnl: dte_pnl[tdte] += t[-1]
        dte_pnl_list = [round(dte_pnl[d], 2) for d in (5, 4, 3, 2, 1)]

        return [code, bt.index, start_time, end_time, variant, short_om, wing_width, sl, dte1re, dte2re, dte3re, dte4re, dte5re, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time, future_price] + trades[0] + re_trades + [total_pnl, re_count] + dte_pnl_list

    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, variant, short_om, wing_width, sl])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            dte_file = get_dte_file(pickle_path)
            max_re = 10

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_Variant/P_ShortOM/P_WingWidth/P_SL/P_Dte1Re/P_Dte2Re/P_Dte3Re/P_Dte4Re/P_Dte5Re/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/T{r}.Time/T{r}.Legs/T{r}.Credit/T{r}.No.Tail.Risk/T{r}.SL.Hit/T{r}.SL.Time/T{r}.PNL'
            log_cols += '/Total.PNL/Re.Count/Dte5.PNL/Dte4.PNL/Dte3.PNL/Dte2.PNL/Dte1.PNL'
            log_cols = log_cols.split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [JADE_LIZARD(wbt, row['entry_time'], row['exit_time'], row['variant'], row['short_om'], row['wing_width'], row['sl'], row['dte1re'], row['dte2re'], row['dte3re'], row['dte4re'], row['dte5re']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))